In [65]:
import torch 
from torch import nn
import torch_geometric
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt
import networkx as nx
from torch_geometric.utils import to_networkx

In [66]:
dataset = QM9(root="data/QM9")
data = dataset[0]
print(dataset)
device = torch.device("mps" if torch.mps.is_available() else "cpu")

QM9(130831)


In [67]:
dataset.num_classes, dataset.num_node_features, dataset.num_edge_features , len(dataset)

(19, 11, 4, 130831)

In [68]:
train_dataset = dataset[:int(0.8 * len(dataset))]
test_dataset = dataset[int(0.8 * len(dataset)):]

In [69]:
train_dataset, test_dataset

(QM9(104664), QM9(26167))

In [70]:
# from torch.utils.data._utils.collate import default_collate
# def collate_to_device(batch):
#     X, y = default_collate(batch)
#     return X, y

In [71]:
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,  shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,  shuffle=False)

In [72]:
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing

class CustomMPNNLayer(MessagePassing):
    def __init__(self, node_channels, edge_channels, out_channels):
        super().__init__(aggr='add') 
        
        self.msg_mlp = nn.Sequential(
            nn.Linear(2 * node_channels + edge_channels, out_channels),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Linear(out_channels, out_channels)
        )
        
        self.update_mlp = nn.Sequential(
            nn.Linear(node_channels + out_channels, out_channels),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Linear(out_channels, out_channels)
        )

    def forward(self, x, edge_index, edge_attr):
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr):
        tmp = torch.cat([x_i, x_j, edge_attr], dim=1)
        return self.msg_mlp(tmp)

    def update(self, aggr_out, x):
        tmp = torch.cat([x, aggr_out], dim=1)
        return self.update_mlp(tmp)

In [73]:
dataset.num_node_features, dataset.num_edge_features

(11, 4)

In [74]:
import torch.nn.functional as F
from torch_geometric.nn import global_add_pool

class CustomQM9Model(nn.Module):
    def __init__(self, node_in_dim=11, edge_in_dim=4, hidden_dim=64):
        super().__init__()
        self.node_proj = nn.Linear(node_in_dim, hidden_dim)
        self.conv1 = CustomMPNNLayer(node_channels=hidden_dim,
                                     edge_channels=edge_in_dim,
                                     out_channels=hidden_dim)

        self.conv2 = CustomMPNNLayer(node_channels=hidden_dim,
                                     edge_channels=edge_in_dim,
                                     out_channels=hidden_dim)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, 1)
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.node_proj(x)
        x = F.relu(x)
        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_attr)
        x = F.relu(x)
        graph_emb = global_add_pool(x, batch)

        return self.head(graph_emb).squeeze()

torch.manual_seed(42)
model_mpnn = CustomQM9Model().to(device)

In [75]:
model_mpnn.state_dict()

OrderedDict([('node_proj.weight',
              tensor([[ 0.2305,  0.2503, -0.0706,  0.2770, -0.0661,  0.0608, -0.1468,  0.1771,
                        0.2658, -0.2212,  0.2621],
                      [ 0.0564,  0.2228,  0.0408,  0.1454, -0.0426,  0.2324,  0.0446, -0.1408,
                        0.0769, -0.1389, -0.0354],
                      [-0.1225,  0.2000, -0.2380, -0.1390, -0.0851, -0.1813,  0.0285, -0.2978,
                        0.2723, -0.2561,  0.2328],
                      [ 0.0502, -0.0979,  0.1863,  0.0470,  0.2436,  0.0330, -0.0951,  0.0810,
                       -0.0818,  0.1269,  0.2692],
                      [ 0.1743, -0.1318,  0.1741,  0.0539,  0.1531, -0.1838, -0.2985, -0.1165,
                       -0.2313,  0.2474,  0.0868],
                      [ 0.1249,  0.0954, -0.0052,  0.2360, -0.2142,  0.0190, -0.2058,  0.0930,
                       -0.1038,  0.0924, -0.0628],
                      [ 0.2501, -0.1787, -0.1798, -0.1798,  0.2712,  0.1005,  0.2901, -0.2

In [76]:
unique_params = sum(dict((p.data_ptr(), p.numel()) for p in model_mpnn.parameters()).values())
unique_params

54081